<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/LightBGM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import yfinance as yf

# 股票代號與名稱對應字典（以部分熱門標的為例）
stock_dict = {
      # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股 (DRAM、Flash、模組、控制晶片)】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO / 磷化銦(InP) 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝製程 / 設備概念股 (含 CoWoS、先進封裝)】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 其他電腦週邊與消費電子概念】 ---
    "2353.TW": "宏碁",
    # --- 【22. MLCC (積層陶瓷電容) 概念股】 ---
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    # --- 【23. 金融股】 ---
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    # --- 【24. 食品與零售概念股】 ---
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    # --- 【25. 力積電與成熟製程 / 特殊晶圓代工概念股】 ---
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TW": "漢磊",
    "3016.TW": "嘉晶",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理"""
  # 複製防汙染
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  # 5日收盤價線性回歸斜率
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)  # 避免除以零
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  # 跳空缺口
  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  # NATR (以 14 日 ATR 為例 / Close)
  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  # 布林頻寬 ((上軌 - 下軌) / 中軌)
  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  # 5日均線乖離率 (BIAS)
  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  # 近似 VWAP 乖離率
  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  # 能量潮 OBV 斜率
  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  # 週轉率
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  # 嚴格防洩漏：特徵全部向後位移 1 期
  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟一：正在下載大盤與個股資料，並為每支股票獨立訓練模型...")

# 下載大盤資料作為相對強度基準[span_2](start_span)[span_2](end_span)
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

# 用來存放每支股票訓練好的模型與評估結果
trained_models = {}

for ticker, name in stock_dict.items():
  try:
    print(f"\n" + "=" * 50)
    print(f" 正在處理標的：{name} ({ticker}) ")
    print("=" * 50)

    df = yf.download(ticker, period="3y", progress=False)
    if df.empty or len(df) < 300:
      print(f"{name} 資料不足，跳過訓練。")
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 目標變數定義：t+1 至 t+5 的 High 是否任一交易日 >= Close(t) * 1.05[span_3](start_span)[span_3](end_span)
    future_high_max = (
        pd.concat([df["High"].shift(-i) for i in range(1, 6)], axis=1)
        .max(axis=1)
    )
    target_condition = future_high_max >= (df["Close"] * 1.05)
    df["Target"] = target_condition.astype(int)

    # 執行特徵工程[span_4](start_span)[span_4](end_span)
    df_feat, feature_cols = compute_features(df, market_df)

    # dropna 清理因 rolling 與 shift 產生的 NaN[span_5](start_span)[span_5](end_span)
    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 100:
      print(f"{name} 清理後資料不足，跳過訓練。")
      continue

    # 依時間順序切割前 80% 訓練、後 20% 測試[span_6](start_span)[span_6](end_span)
    split_idx = int(len(df_clean) * 0.8)
    train_part = df_clean.iloc[:split_idx]
    test_part = df_clean.iloc[split_idx:]

    X_train = train_part[feature_cols].copy()
    y_train = train_part["Target"].copy()
    X_test = test_part[feature_cols].copy()
    y_test = test_part["Target"].copy()

    # 極端值處理 (Winsorization 1% 至 99%) 針對該股票獨立計算分位數[span_7](start_span)[span_7](end_span)
    for col in X_train.columns:
      lower_bound = X_train[col].quantile(0.01)
      upper_bound = X_train[col].quantile(0.99)
      X_train[col] = X_train[col].clip(lower_bound, upper_bound)
      X_test[col] = X_test[col].clip(lower_bound, upper_bound)

    print(
        f"訓練集樣本數: {len(X_train)}, 測試集樣本數: {len(X_test)}, 正樣本比例:"
        f" {y_train.mean():.4f}"
    )

    # 步驟二：為當前股票建立專屬的 LightGBM 模型並訓練[span_8](start_span)[span_8](end_span)
    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=5,
        is_unbalance=True,  # 處理類別不平衡[span_9](start_span)[span_9](end_span)
        random_state=42,
        verbose=-1,
    )

    model.fit(X_train, y_train)
    trained_models[ticker] = model  # 儲存模型字典

    # 步驟三：模型評估與輸出[span_10](start_span)[span_10](end_span)
    y_pred = model.predict(X_test)

    print(f"\n--- {name} ({ticker}) 模型績效評估報告 (測試集) ---")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # 輸出前 15 名特徵重要性[span_11](start_span)[span_11](end_span)
    feature_importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": model.feature_importances_,
    }).sort_values(by="Importance", ascending=False)

    print(f"\n--- {name} ({ticker}) 特徵重要性前 15 名 ---")
    print(feature_importance_df.head(15).to_markdown(index=False))

  except Exception as e:
    print(f"處理 {ticker} 時發生錯誤: {e}")
    continue

print(f"\n所有獨立模型訓練完成！總共成功訓練了 {len(trained_models)} 個股票模型。")


步驟一：正在下載大盤與個股資料，並為每支股票獨立訓練模型...


/tmp/ipykernel_549/107785350.py:331: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_df = yf.download("^TWII", period="3y", progress=False)



 正在處理標的：廣達 (2382.TW) 


/tmp/ipykernel_549/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)


訓練集樣本數: 532, 測試集樣本數: 134, 正樣本比例: 0.3496

--- 廣達 (2382.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.6747    0.6588    0.6667        85
           1     0.4314    0.4490    0.4400        49

    accuracy                         0.5821       134
   macro avg     0.5530    0.5539    0.5533       134
weighted avg     0.5857    0.5821    0.5838       134

Confusion Matrix:
[[56 29]
 [27 22]]

--- 廣達 (2382.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| BB_Bandwidth       |          332 |
| OBV_Slope          |          193 |
| Turnover_Rate      |          192 |
| Volume_Explosion   |          187 |
| NATR               |          186 |
| Close_Slope        |          180 |
| BIAS_5             |          137 |
| Upper_Shadow_Ratio |          134 |
| Gap                |           99 |
| Alpha_5d           |           95 |
| Lower_Shadow_Ratio |           77 |
| VWAP_BIAS          |         

...

訓練集樣本數: 532, 測試集樣本數: 134, 正樣本比例: 0.3534

--- 力積電 (6770.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.5385    0.1667    0.2545        42
           1     0.7107    0.9348    0.8075        92

    accuracy                         0.6940       134
   macro avg     0.6246    0.5507    0.5310       134
weighted avg     0.6567    0.6940    0.6342       134

Confusion Matrix:
[[ 7 35]
 [ 6 86]]

--- 力積電 (6770.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| OBV_Slope          |          274 |
| BB_Bandwidth       |          247 |
| NATR               |          231 |
| Gap                |          222 |
| Turnover_Rate      |          204 |
| Upper_Shadow_Ratio |          179 |
| Close_Slope        |          147 |
| Volume_Explosion   |          144 |
| Lower_Shadow_Ratio |          123 |
| Alpha_5d           |          111 |
| VWAP_BIAS          |           91 |
| BIAS_5             |       

/tmp/ipykernel_549/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)


訓練集樣本數: 532, 測試集樣本數: 134, 正樣本比例: 0.3120

--- 茂矽 (2342.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.3333    0.2889    0.3095        45
           1     0.6632    0.7079    0.6848        89

    accuracy                         0.5672       134
   macro avg     0.4982    0.4984    0.4972       134
weighted avg     0.5524    0.5672    0.5588       134

Confusion Matrix:
[[13 32]
 [26 63]]

--- 茂矽 (2342.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| NATR               |          300 |
| Gap                |          242 |
| BB_Bandwidth       |          234 |
| OBV_Slope          |          229 |
| Alpha_5d           |          184 |
| Upper_Shadow_Ratio |          178 |
| Turnover_Rate      |          178 |
| Volume_Explosion   |          166 |
| Lower_Shadow_Ratio |          130 |
| BIAS_5             |          110 |
| VWAP_BIAS          |          102 |
| Close_Slope        |         

/tmp/ipykernel_549/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 3707.TW"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['3707.TW']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')
/tmp/ipykernel_549/107785350.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="3y", progress=False)


漢磊 資料不足，跳過訓練。

 正在處理標的：嘉晶 (3016.TW) 
訓練集樣本數: 532, 測試集樣本數: 134, 正樣本比例: 0.4041

--- 嘉晶 (3016.TW) 模型績效評估報告 (測試集) ---
              precision    recall  f1-score   support

           0     0.2353    0.1026    0.1429        39
           1     0.7009    0.8632    0.7736        95

    accuracy                         0.6418       134
   macro avg     0.4681    0.4829    0.4582       134
weighted avg     0.5654    0.6418    0.5900       134

Confusion Matrix:
[[ 4 35]
 [13 82]]

--- 嘉晶 (3016.TW) 特徵重要性前 15 名 ---
| Feature            |   Importance |
|:-------------------|-------------:|
| NATR               |          263 |
| BB_Bandwidth       |          219 |
| OBV_Slope          |          197 |
| Gap                |          179 |
| VWAP_BIAS          |          177 |
| Turnover_Rate      |          173 |
| Alpha_5d           |          164 |
| Close_Slope        |          159 |
| Upper_Shadow_Ratio |          151 |
| Volume_Explosion   |          137 |
| Lower_Shadow_Ratio |          

In [2]:
#保留均線個股獨立模型訓練
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import yfinance as yf

# 股票代號與名稱對應字典 (完整保留股票資料池)
stock_dict = {
   # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股 (DRAM、Flash、模組、控制晶片)】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO / 磷化銦(InP) 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝製程 / 設備概念股 (含 CoWoS、先進封裝)】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 其他電腦週邊與消費電子概念】 ---
    "2353.TW": "宏碁",
    # --- 【22. MLCC (積層陶瓷電容) 概念股】 ---
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    # --- 【23. 金融股】 ---
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    # --- 【24. 食品與零售概念股】 ---
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    # --- 【25. 力積電與成熟製程 / 特殊晶圓代工概念股】 ---
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    # --- 【26. 半導體廠務工程與無塵室機電概念股】 ---
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "5536.TWO": "聖暉*",
    "6613.TWO": "朋億*",
    "6691.TW": "信紘科",
    # --- 【27. 台積電擴廠 / 先進製程特化與關鍵供應鏈概念股】 ---
    "4755.TW": "三福化",
    "1773.TW": "勝一",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    # --- 【28. 精密化學與長興材料概念股】 ---
    "1717.TW": "長興",
    # --- 【29. 玻纖布與PCB上游材料概念股】 ---
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    # --- 【30. PCB與半導體自動化及鑽孔設備概念股】 ---
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    # --- 【31. 驅動IC封測與COF基板相關概念股】 ---
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    # --- 【32. 銲錫與電子材料 / 錫膏相關概念股】 ---
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    # --- 【33. 伺服器滑軌與精密機構件概念股 (川湖、南俊國際等)】 ---
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print("步驟一：正在篩選符合流動性與均線糾結條件的股票...")
qualified_tickers = []

for ticker, name in stock_dict.items():
  try:
    df = yf.download(ticker, period="2mo", progress=False)
    if df.empty or len(df) < 20:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    df["5MA"] = df["Close"].rolling(window=5).mean()
    df["10MA"] = df["Close"].rolling(window=10).mean()
    df["20MA"] = df["Close"].rolling(window=20).mean()
    df["5MA_Vol"] = df["Volume"].rolling(window=5).mean()

    ma5 = df["5MA"].iloc[-1]
    ma10 = df["10MA"].iloc[-1]
    ma20 = df["20MA"].iloc[-1]
    vol_5ma = df["5MA_Vol"].iloc[-1]

    if pd.isna(ma5) or pd.isna(ma10) or pd.isna(ma20) or pd.isna(vol_5ma):
      continue

    # 保留篩選條件：5日均量 >= 500張 且 均線糾結絕對值 < 3%[span_0](start_span)[span_0](end_span)
    if (
        vol_5ma >= 500000
        and abs((ma5 - ma20) / ma20) < 0.03
        and abs((ma10 - ma20) / ma20) < 0.03
    ):
      qualified_tickers.append((ticker, name))
  except Exception:
    pass

print(f"篩選完成，共找到 {len(qualified_tickers)} 檔符合條件的標的。")

if len(qualified_tickers) == 0:
  print("今日盤勢中暫無符合該嚴格條件的標的。")
else:
  print("步驟二：正在分別對每檔合格股票套用進階特徵工程與獨立模型訓練預測...")
  predictions = []

  for ticker, name in qualified_tickers:
    try:
      # 下載每檔股票的 2 年歷史資料[span_1](start_span)[span_1](end_span)
      df = yf.download(ticker, period="2y", progress=False)
      if df.empty or len(df) < 250:
        continue
      if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

      # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%[span_2](start_span)[span_2](end_span)
      horizon = 10
      threshold = 0.10
      future_max = (
          df["Close"]
          .shift(-1)
          .rolling(window=horizon, min_periods=1)
          .max()
          .shift(-(horizon - 1))
      )
      future_return = (future_max - df["Close"]) / df["Close"]
      df["Target"] = (future_return >= threshold).astype(int)

      # 執行四大類特徵工程與防洩漏處理[span_3](start_span)[span_3](end_span)
      df_feat, feature_cols = compute_features(df, market_df)

      df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
      if len(df_clean) < 50:
        continue

      # 訓練資料使用歷史資料（排除最後一筆正在預測當下的資料）[span_4](start_span)[span_4](end_span)
      X_train = df_clean[feature_cols].iloc[:-1]
      y_train = df_clean["Target"].iloc[:-1]

      if len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)[span_5](start_span)[span_5](end_span)
      for col in X_train.columns:
        lower_bound = X_train[col].quantile(0.01)
        upper_bound = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower_bound, upper_bound)

      train_base = y_train.mean()

      # 針對該股票訓練專屬的獨立隨機森林模型[span_6](start_span)[span_6](end_span)
      model = RandomForestClassifier(
          n_estimators=100, max_depth=5, random_state=42
      )
      model.fit(X_train, y_train)

      # 取得該股票最新一筆特徵進行預測[span_7](start_span)[span_7](end_span)
      latest_features = df_clean[feature_cols].iloc[[-1]].copy()
      for col in latest_features.columns:
        lb = X_train[col].quantile(0.01)
        ub = X_train[col].quantile(0.99)
        latest_features[col] = latest_features[col].clip(lb, ub)

      if latest_features.dropna().empty:
        continue

      prob = model.predict_proba(latest_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0

      predictions.append({
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "y成立機率值": round(float(prob), 4),
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
    except Exception:
      pass

  final_output_df = pd.DataFrame(predictions)
  if not final_output_df.empty:
    print("\n" + "=" * 65)
    print(" 依個股獨立模型預測之結果與機率值清單 (保留篩選條件) ")
    print("=" * 65)
    print(
        final_output_df[[
            "股票名稱",
            "股票代號",
            "Base y",
            "y成立機率值",
            "10% Lift 值",
        ]].to_markdown(index=False)
    )
  else:
    print("目前無法產生預測結果。")

步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：正在篩選符合流動性與均線糾結條件的股票...


/tmp/ipykernel_549/2032543599.py:352: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_df = yf.download("^TWII", period="3y", progress=False)
/tmp/ipykernel_549/2032543599.py:361: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_549/2032543599.py:361: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_549/2032543599.py:361: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_549/2032543599.py:361: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2mo", progress=False)
/tmp/ipykernel_549/2032543599.py:361: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.do

篩選完成，共找到 35 檔符合條件的標的。
步驟二：正在分別對每檔合格股票套用進階特徵工程與獨立模型訓練預測...


/tmp/ipykernel_549/2032543599.py:401: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/2032543599.py:401: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/2032543599.py:401: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/2032543599.py:401: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/2032543599.py:401: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/2032543599.py:401: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticke


 依個股獨立模型預測之結果與機率值清單 (保留篩選條件) 
| 股票名稱   |   股票代號 | Base y   |   y成立機率值 | 10% Lift 值   |
|:-----------|-----------:|:---------|--------------:|:--------------|
| 廣達       |       2382 | 10.14%   |        0.1573 | 1.55x         |
| 神達       |       3706 | 25.24%   |        0.2192 | 0.87x         |
| 神基       |       3005 | 13.44%   |        0.1911 | 1.42x         |
| 佳必琪     |       6197 | 23.58%   |        0.2425 | 1.03x         |
| 中磊       |       5388 | 10.61%   |        0.5338 | 5.03x         |
| 合勤控     |       3704 | 18.63%   |        0.4803 | 2.58x         |
| 譜瑞-KY    |       4966 | 15.8%    |        0.0905 | 0.57x         |
| 祥碩       |       5269 | 17.92%   |        0.1452 | 0.81x         |
| 嘉澤       |       3533 | 24.06%   |        0.4666 | 1.94x         |
| 優群       |       3217 | 15.57%   |        0.1176 | 0.76x         |
| 智原       |       3035 | 18.4%    |        0.1314 | 0.71x         |
| 群電       |       6412 | 13.44%   |        0.0654 | 0.49x         |
| 亞力       |   

In [6]:
#移除均線糾結隨機森林預測
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import yfinance as yf

# 股票代號與名稱對應字典
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21-33. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "5536.TWO": "聖暉*",
    "6613.TWO": "朋億*",
    "6691.TW": "信紘科",
    "4755.TW": "三福化",
    "1773.TW": "勝一",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    # --- 【34. 銅箔與PCB上游材料】 ---
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (已移除均線糾結條件)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股處理，並產出最近 5 個交易日的逐日獨立模型預測..."
)
predictions = []

for ticker, name in stock_dict.items():
  try:
    df = yf.download(ticker, period="2y", progress=False)
    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測 (使用至當天為止的資料)
    for i in range(-5, 0):
      # 訓練集：嚴格使用索引小於當前目標日期的資料，防止未來資訊外洩
      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 訓練隨機森林模型
      model = RandomForestClassifier(
          n_estimators=100, max_depth=5, random_state=42
      )
      model.fit(X_train_clipped, y_train)

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = df_clean.index[i].strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),  # 用於排序的原始數值
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",  # 轉為百分比顯示
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  # 依「預測日期」由新到舊排序，同日期再依「預測機率」由大到小排序
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(" 最近 5 個交易日個股逐日預測結果清單 (依日期與機率排序) ")
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前無法產生預測結果。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股處理，並產出最近 5 個交易日的逐日獨立模型預測...


/tmp/ipykernel_549/989168442.py:344: FutureWarning: YF.download() has changed argument auto_adjust default to True
  market_df = yf.download("^TWII", period="3y", progress=False)
/tmp/ipykernel_549/989168442.py:355: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/989168442.py:355: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/989168442.py:355: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/989168442.py:355: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="2y", progress=False)
/tmp/ipykernel_549/989168442.py:355: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tic


 最近 5 個交易日個股逐日預測結果清單 (依日期與機率排序) 
| 預測日期   | 股票名稱    |   股票代號 | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:------------|-----------:|:---------|:--------------|:--------------|
| 2026-08-14 | 大量        |       3167 | 49.64%   | 75.16%        | 1.51x         |
| 2026-08-14 | 愛普*       |       6531 | 36.34%   | 74.21%        | 2.04x         |
| 2026-08-14 | 國巨        |       2327 | 30.88%   | 69.99%        | 2.27x         |
| 2026-08-14 | 頎邦        |       6147 | 18.53%   | 68.18%        | 3.68x         |
| 2026-08-14 | 禾伸堂      |       3026 | 28.98%   | 66.52%        | 2.3x          |
| 2026-08-14 | 萬潤        |       6187 | 34.2%    | 66.12%        | 1.93x         |
| 2026-08-14 | 金居        |       8358 | 43.71%   | 64.75%        | 1.48x         |
| 2026-08-14 | 群聯        |       8299 | 41.33%   | 63.79%        | 1.54x         |
| 2026-08-14 | 台燿        |       6274 | 44.18%   | 62.27%        | 1.41x         |
| 2026-08-14 | 宏致        |       3605 | 33.02%   | 61.2%         | 1

In [7]:
#無均線糾結升級為lightgbm優化模式
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21-33. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "5536.TWO": "聖暉*",
    "6613.TWO": "朋億*",
    "6691.TW": "信紘科",
    "4755.TW": "三福化",
    "1773.TW": "勝一",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    # --- 【34. 銅箔與PCB上游材料】 ---
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，並產出最近 5 個交易日的逐日獨立模型預測..."
)
predictions = []

# 批次下載所有股票資料以大幅提升速度
all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="2y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測
    for i in range(-5, 0):
      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 切分出驗證集供 LightGBM 早停機制使用
      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      # 建立 LightGBM 模型並加入早停機制
      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )

      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = df_clean.index[i].strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(" 最近 5 個交易日個股逐日預測結果清單 (LightGBM + 早停機制優化版) ")
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前無法產生預測結果。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，並產出最近 5 個交易日的逐日獨立模型預測...

 最近 5 個交易日個股逐日預測結果清單 (LightGBM + 早停機制優化版) 
| 預測日期   | 股票名稱    |   股票代號 | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:------------|-----------:|:---------|:--------------|:--------------|
| 2026-08-14 | 信驊        |       5274 | 40.8%    | 71.13%        | 1.74x         |
| 2026-08-14 | 禾伸堂      |       3026 | 29.01%   | 68.89%        | 2.37x         |
| 2026-08-14 | 頎邦        |       6147 | 18.63%   | 65.09%        | 3.49x         |
| 2026-08-14 | 大銀微系統  |       4576 | 31.37%   | 62.13%        | 1.98x         |
| 2026-08-14 | 南電        |       8046 | 44.58%   | 62.09%        | 1.39x         |
| 2026-08-14 | 鈺創        |       5351 | 36.56%   | 57.51%        | 1.57x         |
| 2026-08-14 | 德宏        |       5475 | 50.71%   | 55.09%        | 1.09x         |
| 2026-08-14 | 聯亞        |       3081 | 51.18%   | 54.22%        | 1.06x         |
| 2026-08-14 | AES-KY      |       6781 | 33.73%   | 54.09%        | 1.6x  

In [8]:
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典
stock_dict = {
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21-33. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "2404.TW": "漢唐",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "5536.TWO": "聖暉*",
    "6613.TWO": "朋億*",
    "6691.TW": "信紘科",
    "4755.TW": "三福化",
    "1773.TW": "勝一",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    # --- 【34. 銅箔與PCB上游材料】 ---
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，並套用流動性與均線糾結篩選條件..."
)
predictions = []

all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="2y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 計算篩選條件所需的指標（直接使用當天實際數據，不作 shift）
    vol_mean_5 = df["Volume"].rolling(5).mean()
    ma5 = df["Close"].rolling(5).mean()
    ma10 = df["Close"].rolling(10).mean()
    ma20 = df["Close"].rolling(20).mean()
    ma_max = pd.concat([ma5, ma10, ma20], axis=1).max(axis=1)
    ma_min = pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)

    # 條件 1：5日均量 >= 500 張 (500,000 股)
    liquidity_ok = vol_mean_5 >= 500000
    # 條件 2：5日線、10日線、20日線差距在 3% 以內
    ma_tangle = (ma_max - ma_min) / (ma_min + 1e-6) <= 0.03

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測
    for i in range(-5, 0):
      # 取得當前預測日期的 Index 標籤
      target_idx = df_clean.index[i]

      # 套用篩選條件：若當天不符合流動性或均線糾結，則略過
      if not liquidity_ok.loc[target_idx] or not ma_tangle.loc[target_idx]:
        continue

      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 切分出驗證集供 LightGBM 早停機制使用
      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      # 建立 LightGBM 模型並加入早停機制
      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )

      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = target_idx.strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(
      " 最近 5 個交易日符合篩選條件之個股逐日預測結果清單 (流動性 & 均線糾結過濾版)"
  )
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前最近 5 個交易日內，沒有符合「5日均量 >= 500張 且 均線糾結 3%」的股票。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，並套用流動性與均線糾結篩選條件...

 最近 5 個交易日符合篩選條件之個股逐日預測結果清單 (流動性 & 均線糾結過濾版)
| 預測日期   | 股票名稱   |   股票代號 | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:-----------|-----------:|:---------|:--------------|:--------------|
| 2026-08-14 | 臻鼎-KY    |       4958 | 31.84%   | 41.78%        | 1.31x         |
| 2026-08-14 | 矽格       |       6257 | 25.47%   | 40.29%        | 1.58x         |
| 2026-08-14 | 鈦昇       |       8027 | 29.48%   | 35.26%        | 1.2x          |
| 2026-08-14 | 辛耘       |       3583 | 25.47%   | 33.43%        | 1.31x         |
| 2026-08-14 | 華泰       |       2329 | 28.77%   | 33.02%        | 1.15x         |
| 2026-08-14 | 新應材     |       4749 | 27.59%   | 29.16%        | 1.06x         |
| 2026-08-14 | 立敦       |       6175 | 30.66%   | 28.89%        | 0.94x         |
| 2026-08-14 | 嘉澤       |       3533 | 24.06%   | 28.23%        | 1.17x         |
| 2026-08-14 | 京元電子   |       2449 | 32.08%   | 27.45%        | 0.86x         |
| 2026-